In [1]:
import sys
import os

parent_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))

if parent_dir not in sys.path:
    sys.path.append(parent_dir)
    
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

### Model Training, ML pipeline construction, Hyperparameter Tuning, and Model Evaluation

- 3 models: Polynomial regression, Decision Tree, Random Forest
- Random forest was chosen as final model with lowest RMSE, MAPE error
- Hyperparameter tuning for model complexity with grid search

In [2]:
train_set = pd.read_csv("../data/train_set.csv")
test_set = pd.read_csv("../data/test_set.csv")
print(f"Shape: Train set {train_set.shape}, Test set {test_set.shape}")

Shape: Train set (15854, 10), Test set (3964, 9)


In [3]:
X_train = train_set.drop(columns=['transaction_price', 'year', 'scheme_name_area'])
y_train = train_set['transaction_price']

In [4]:
from sklearn.preprocessing import PolynomialFeatures, StandardScaler, FunctionTransformer, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

In [5]:
X_train.head()

,property_type,mukim,tenure,land_parcel_area,unit_level,income_median_area,area_median_price
0,Condominium/Apartment,Mukim Kuala Lumpur,Leasehold,1074.130619,7,11221.583852,465000.0
1,Flat,Mukim Batu,Leasehold,764.237638,4,11131.878211,185000.0
2,Condominium/Apartment,Kuala Lumpur Town Centre,Freehold,4200.000000,36,10302.979398,565000.0
3,Condominium/Apartment,Mukim Batu,Freehold,861.112832,19,10518.458089,475000.0
4,Condominium/Apartment,Mukim Setapak,Leasehold,832.050274,12,9519.480524,214000.0


In [6]:
train_set.head()

,property_type,mukim,scheme_name_area,tenure,land_parcel_area,unit_level,transaction_price,year,income_median_area,area_median_price
0,Condominium/Apartment,Mukim Kuala Lumpur,MIDFIELDS SG.BESI KONDOMINIUM,Leasehold,1074.130619,7,420000,2021,11221.583852,465000.0
1,Flat,Mukim Batu,TMN SRI MURNI MEDIUM COST FLAT,Leasehold,764.237638,4,190000,2023,11131.878211,185000.0
2,Condominium/Apartment,Kuala Lumpur Town Centre,Others,Freehold,4200.000000,36,2300000,2022,10302.979398,565000.0
3,Condominium/Apartment,Mukim Batu,RESIDENSI SELINGSING,Freehold,861.112832,19,470000,2022,10518.458089,475000.0
4,Condominium/Apartment,Mukim Setapak,GENTING COURT,Leasehold,832.050274,12,250000,2021,9519.480524,214000.0


In [7]:
test_set.head()

,property_type,mukim,scheme_name_area,tenure,land_parcel_area,unit_level,transaction_price,year,income_median_area
0,Condominium/Apartment,Kuala Lumpur Town Centre,PARK SEVEN,Freehold,4200.000000,8,4400000,2025,10655.952127
1,Flat,Mukim Ampang,DESA PANDAN,Leasehold,721.181997,4,300000,2022,10231.924082
2,Condominium/Apartment,Mukim Petaling,RESIDENSI HAMSTEAD,Leasehold,947.008837,16,411000,2024,10346.591315
3,Condominium/Apartment,Kuala Lumpur Town Centre,PANGSAPURI BANDAR BARU SENTUL,Leasehold,559.723341,5,150000,2022,10302.979398
4,Condominium/Apartment,Mukim Kuala Lumpur,PRISMA PERDANA APT,Freehold,818.057190,6,300000,2023,11404.069649


In [8]:
num_cols = ['land_parcel_area', 'unit_level', 'income_median_area', 'area_median_price']
cat_cols = ['property_type', 'mukim', 'tenure']

def preprocess_linear(d=1):
    log_transform = ColumnTransformer([
        ("log_transform",
         FunctionTransformer(np.log, feature_names_out="one-to-one"),
         ['land_parcel_area', 'area_median_price'])
    ], remainder='passthrough')

    num_pipeline = Pipeline([
        ("log", log_transform),
        ("poly", PolynomialFeatures(degree=d, include_bias=False)),
        ("scaler", StandardScaler())
    ])

    preprocess = ColumnTransformer([
        ("num_pipeline", num_pipeline, num_cols),
        ("cat_encode", OneHotEncoder(), cat_cols)
    ])

    return preprocess

In [9]:
preprocess = preprocess_linear()
X_train_lin = preprocess.fit_transform(X_train)
preprocess.get_feature_names_out()

array(['num_pipeline__log_transform__land_parcel_area',
       'num_pipeline__log_transform__area_median_price',
       'num_pipeline__remainder__unit_level',
       'num_pipeline__remainder__income_median_area',
       'cat_encode__property_type_Condominium/Apartment',
       'cat_encode__property_type_Flat',
       'cat_encode__property_type_Low-Cost Flat',
       'cat_encode__property_type_Town House',
       'cat_encode__mukim_Kuala Lumpur Town Centre',
       'cat_encode__mukim_Mukim Ampang', 'cat_encode__mukim_Mukim Batu',
       'cat_encode__mukim_Mukim Cheras',
       'cat_encode__mukim_Mukim Kuala Lumpur',
       'cat_encode__mukim_Mukim Petaling',
       'cat_encode__mukim_Mukim Setapak',
       'cat_encode__mukim_Mukim Ulu Kelang',
       'cat_encode__tenure_Freehold', 'cat_encode__tenure_Leasehold'],
      dtype=object)

In [10]:
cat_cols = ['property_type', 'mukim', 'tenure']

def preprocess_tree():
    log_pipeline = Pipeline([
        ("log_transform", FunctionTransformer(np.log, feature_names_out="one-to-one")),
        ("scaler", StandardScaler())
    ])

    preprocess = ColumnTransformer([
        ("log", log_pipeline, ['land_parcel_area', 'area_median_price']),
        ("scaler", StandardScaler(), ['unit_level', 'income_median_area']),
        ("cat_encode", OneHotEncoder(), cat_cols)
    ])
    
    return preprocess

preprocess = preprocess_tree()
X_train_prep = preprocess.fit_transform(X_train)
preprocess.get_feature_names_out()

array(['log__land_parcel_area', 'log__area_median_price',
       'scaler__unit_level', 'scaler__income_median_area',
       'cat_encode__property_type_Condominium/Apartment',
       'cat_encode__property_type_Flat',
       'cat_encode__property_type_Low-Cost Flat',
       'cat_encode__property_type_Town House',
       'cat_encode__mukim_Kuala Lumpur Town Centre',
       'cat_encode__mukim_Mukim Ampang', 'cat_encode__mukim_Mukim Batu',
       'cat_encode__mukim_Mukim Cheras',
       'cat_encode__mukim_Mukim Kuala Lumpur',
       'cat_encode__mukim_Mukim Petaling',
       'cat_encode__mukim_Mukim Setapak',
       'cat_encode__mukim_Mukim Ulu Kelang',
       'cat_encode__tenure_Freehold', 'cat_encode__tenure_Leasehold'],
      dtype=object)

In [11]:
from sklearn.linear_model import Ridge
from sklearn.compose import TransformedTargetRegressor
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error, mean_absolute_percentage_error

In [12]:
def make_model_linear(d=1, reg=1):
    
    preprocessing = preprocess_linear(d)
    target_transform = TransformedTargetRegressor(
        regressor=Ridge(alpha=reg),
        transformer=FunctionTransformer(np.log, inverse_func=np.exp)
    )

    model = Pipeline([
        ("preprocess", preprocessing),
        ("model", target_transform)
    ])

    return model

In [13]:
model_lin = make_model_linear()

model_lin.fit(X_train, y_train)
y_hat = model_lin.predict(X_train)
print(int(root_mean_squared_error(y_train, y_hat)))
print(y_hat[:5].round())
print(y_train[:5].to_numpy())

333819
[ 484126.  184614. 3235675.  413047.  258116.]
[ 420000  190000 2300000  470000  250000]


In [14]:
degree_list = [1, 2, 3, 4, 5, 6]
reg_list = np.linspace(0.1, 5, 50)

params = {
    "preprocess__num_pipeline__poly__degree":degree_list,
    "model__regressor__alpha":reg_list
}

search_lin = GridSearchCV(model_lin, params, scoring='neg_root_mean_squared_error')
search_lin.fit(X_train, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...nc 'exp'>)))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__regressor__alpha': array([0.1, 0....8, 4.9, 5. ]), 'preprocess__num_pipeline__poly__degree': [1, 2, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_root_mean_squared_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",None
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for ea

In [15]:
search_lin.best_params_

{'model__regressor__alpha': np.float64(0.2),
 'preprocess__num_pipeline__poly__degree': 5}

In [16]:
search_lin.best_score_

np.float64(-273286.41822411394)

In [17]:
cv_res = pd.DataFrame(search_lin.cv_results_)
cv_res.sort_values(by='rank_test_score')[:5]

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_model__regressor__alpha,param_preprocess__num_pipeline__poly__degree,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
10,0.077461,0.021030,0.009998,0.001069,0.2,5,"{'model__regressor__alpha': 0.2, 'preprocess__...",-277143.033845,-279358.001357,-288492.692899,-255400.179206,-266038.183814,-273286.418224,11449.216948,1
29,0.091534,0.000266,0.012267,0.004177,0.5,6,"{'model__regressor__alpha': 0.5, 'preprocess__...",-277243.895065,-278321.198160,-290316.009939,-255151.315600,-265828.520126,-273372.187778,11962.299403,2
35,0.092232,0.006419,0.014120,0.005068,0.6,6,"{'model__regressor__alpha': 0.6, 'preprocess__...",-277043.760925,-279113.054087,-289951.025218,-255318.215557,-265845.695009,-273454.350159,11864.704794,3
23,0.101073,0.008051,0.011828,0.003258,0.4,6,"{'model__regressor__alpha': 0.4, 'preprocess__...",-277620.149684,-277448.506735,-290973.483469,-255306.728996,-265987.154635,-273467.204704,12044.264536,4
41,0.094687,0.003303,0.013823,0.003925,0.7,6,{'model__regressor__alpha': 0.7000000000000001...,-276941.050073,-279833.647883,-289747.481490,-255658.566749,-265958.945170,-273627.938273,11756.181211,5


In [18]:
def make_model_tree(depth=None, min_samples=2):
    
    preprocessing = preprocess_tree()
    target_transform = TransformedTargetRegressor(
        regressor=DecisionTreeRegressor(max_depth=depth, min_samples_split=min_samples, random_state=1),
        transformer=FunctionTransformer(np.log, inverse_func=np.exp)
    )

    model = Pipeline([
        ("preprocess", preprocessing),
        ("model", target_transform)
    ])

    return model

In [19]:
model_tree = make_model_tree()

model_tree.fit(X_train, y_train)
y_hat = model_tree.predict(X_train)
print(root_mean_squared_error(y_train, y_hat))

38077.221283238425


In [20]:
pd.Series(-cross_val_score(model_tree, X_train, y_train, scoring='neg_root_mean_squared_error', cv=5)).describe()

count         5.000000
mean     252269.670217
std       19341.385432
min      224729.489127
25%      246658.240371
50%      248501.634862
75%      268418.778024
max      273040.208701
dtype: float64

In [21]:
max_depth_list = [2, 4, 8, 16, 32, 64, None]
min_samples_split_list = [2, 10, 20, 30, 50, 100, 200, 300, 500]

model_tree = make_model_tree()

params = {
    "model__regressor__max_depth":max_depth_list,
    "model__regressor__min_samples_split":min_samples_split_list
}

search_tree = GridSearchCV(model_tree, params, scoring='neg_root_mean_squared_error')
search_tree.fit(X_train, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...nc 'exp'>)))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__regressor__max_depth': [2, 4, ...], 'model__regressor__min_samples_split': [2, 10, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_root_mean_squared_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",None
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and param

In [22]:
search_tree.best_params_

{'model__regressor__max_depth': 32, 'model__regressor__min_samples_split': 20}

In [23]:
search_tree.best_score_

np.float64(-219220.655155888)

In [24]:
cv_res_tree = pd.DataFrame(search_tree.cv_results_)
cv_res_tree.sort_values(by='rank_test_score')[:5]

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_model__regressor__max_depth,param_model__regressor__min_samples_split,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
47,0.046123,0.004534,0.005369,0.004992,64,20,"{'model__regressor__max_depth': 64, 'model__re...",-219187.539125,-233468.364594,-225817.982033,-195564.258317,-222065.131711,-219220.655156,12761.184906,1
56,0.046575,0.004822,0.006938,0.000950,None,20,"{'model__regressor__max_depth': None, 'model__...",-219187.539125,-233468.364594,-225817.982033,-195564.258317,-222065.131711,-219220.655156,12761.184906,1
38,0.044360,0.001288,0.007184,0.001931,32,20,"{'model__regressor__max_depth': 32, 'model__re...",-219187.539125,-233468.364594,-225817.982033,-195564.258317,-222065.131711,-219220.655156,12761.184906,1
29,0.043382,0.001880,0.003159,0.002810,16,20,"{'model__regressor__max_depth': 16, 'model__re...",-219368.835810,-233856.335577,-228859.435532,-195883.030506,-221953.231630,-219984.173811,13086.848978,4
28,0.049858,0.001954,0.003881,0.003916,16,10,"{'model__regressor__max_depth': 16, 'model__re...",-219494.005505,-244724.980530,-235545.954738,-205868.521378,-223319.595753,-225790.611581,13388.949869,5


In [26]:
def make_model_forest(n=100, depth=None, min_samples=2):
    
    preprocessing = preprocess_tree()
    target_transform = TransformedTargetRegressor(
        regressor=RandomForestRegressor(n_estimators=n,
                                        max_depth=depth,
                                        min_samples_split=min_samples,
                                        random_state=1),
        transformer=FunctionTransformer(np.log, inverse_func=np.exp)
    )

    model = Pipeline([
        ("preprocess", preprocessing),
        ("model", target_transform)
    ])

    return model

In [ ]:
n_estimators_list = [50, 100, 250, 500]
max_depth_list = [8, 12, 16, 32, None]
min_samples_split_list = [2, 5, 10]

model_forest = make_model_forest()

params = {
    "model__regressor__n_estimators":n_estimators_list,
    "model__regressor__max_depth":max_depth_list,
    "model__regressor__min_samples_split":min_samples_split_list
}

search_forest = GridSearchCV(model_forest, params, scoring='neg_root_mean_squared_error')
search_forest.fit(X_train, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...nc 'exp'>)))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__regressor__max_depth': [8, 16, ...], 'model__regressor__min_samples_split': [2, 5, ...], 'model__regressor__n_estimators': [50, 100, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_root_mean_squared_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",None
,"verbose verbose: intControls the verbosity: the higher, the more messages.-

In [28]:
search_forest.best_params_

{'model__regressor__max_depth': None,
 'model__regressor__min_samples_split': 2,
 'model__regressor__n_estimators': 250}

In [29]:
search_forest.best_score_

np.float64(-190454.3860586147)

In [68]:
cv_res_forest = pd.DataFrame(search_forest.cv_results_)
cv_res_forest.sort_values(by='rank_test_score')[:10]

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_model__regressor__max_depth,param_model__regressor__min_samples_split,param_model__regressor__n_estimators,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
50,7.748977,0.149683,0.177155,0.004185,None,2,250,"{'model__regressor__max_depth': None, 'model__...",-188230.690483,-199466.312555,-197419.303473,-174487.037655,-192668.586127,-190454.386059,8882.889314,1
34,7.599484,0.243541,0.174523,0.009675,32,2,250,"{'model__regressor__max_depth': 32, 'model__re...",-188265.998890,-199550.642897,-197479.140336,-174387.030903,-192650.845595,-190466.731724,8942.725241,2
49,3.253844,0.180181,0.077887,0.005628,None,2,100,"{'model__regressor__max_depth': None, 'model__...",-188818.290806,-198091.651312,-197410.305997,-173979.910613,-194340.666516,-190528.165049,8898.019451,3
38,6.316981,0.124333,0.136127,0.011128,32,5,250,"{'model__regressor__max_depth': 32, 'model__re...",-188651.665580,-199557.640126,-197048.371053,-174358.235569,-193147.669541,-190552.716374,8898.215171,4
54,6.532407,0.389493,0.153178,0.023248,None,5,250,"{'model__regressor__max_depth': None, 'model__...",-188651.665580,-199574.071950,-197046.375386,-174366.297019,-193143.853162,-190556.452619,8898.095858,5
33,2.991796,0.069132,0.068903,0.005038,32,2,100,"{'model__regressor__max_depth': 32, 'model__re...",-188811.479329,-198171.159336,-197414.523702,-174100.592346,-194371.771248,-190573.905192,8870.313407,6
37,2.588758,0.061243,0.057779,0.003889,32,5,100,"{'model__regressor__max_depth': 32, 'model__re...",-188249.943319,-198559.705894,-196979.658419,-174348.008770,-194857.208968,-190598.905074,8852.465950,7
53,2.525204,0.022688,0.063368,0.003397,None,5,100,"{'model__regressor__max_depth': None, 'model__...",-188249.943319,-198576.889894,-196979.658419,-174348.185544,-194871.545033,-190605.244442,8856.873183,8
51,15.243309,0.256206,0.335177,0.010008,None,2,500,"{'model__regressor__max_depth': None, 'model__...",-188219.536995,-200195.476011,-197328.001773,-175497.084901,-191887.339547,-190625.487845,8634.324656,9
35,15.328419,0.167284,0.330240,0.010592,32,2,500,"{'model__regressor__max_depth': 32, 'model__re...",-188242.156521,-200308.496279,-197424.689223,-175405.746051,-191842.448220,-190644.707259,8703.904248,10


In [31]:
forest_best_model = search_forest.best_estimator_
feature_importance = forest_best_model["model"].regressor_.feature_importances_
sorted(zip(feature_importance, forest_best_model["preprocess"].get_feature_names_out()), reverse=True)

[(np.float64(0.6869787731028708), 'log__land_parcel_area'),
 (np.float64(0.12247245781699986),
  'cat_encode__property_type_Condominium/Apartment'),
 (np.float64(0.11600406843881395), 'log__area_median_price'),
 (np.float64(0.03186790799709673), 'scaler__unit_level'),
 (np.float64(0.01881482708325295), 'scaler__income_median_area'),
 (np.float64(0.0031340808418130295), 'cat_encode__tenure_Freehold'),
 (np.float64(0.0028877252864785677), 'cat_encode__tenure_Leasehold'),
 (np.float64(0.0027772382210657837),
  'cat_encode__mukim_Kuala Lumpur Town Centre'),
 (np.float64(0.0027230253543164937), 'cat_encode__mukim_Mukim Setapak'),
 (np.float64(0.0026631817632640337), 'cat_encode__mukim_Mukim Ampang'),
 (np.float64(0.002064356268488777), 'cat_encode__mukim_Mukim Kuala Lumpur'),
 (np.float64(0.001869943381367568), 'cat_encode__property_type_Flat'),
 (np.float64(0.0016359485000664044), 'cat_encode__mukim_Mukim Batu'),
 (np.float64(0.0013935535576294071), 'cat_encode__property_type_Town House'),

In [32]:
area_mapping = train_set.groupby(['mukim', 'scheme_name_area'])['area_median_price'].median().to_dict()

others_value_by_mukim = (
    train_set[train_set['scheme_name_area'] == 'Others']
    .groupby('mukim')['area_median_price']
    .median()
)

others_value = train_set['area_median_price'].median()

def map_test_row(row):
    key = (row['mukim'], row['scheme_name_area'])
    
    if key in area_mapping:
        return area_mapping[key]
    
    return others_value_by_mukim.get(row['mukim'], others_value)

test_set['area_median_price'] = test_set.apply(map_test_row, axis=1)
test_set.head()

,property_type,mukim,scheme_name_area,tenure,land_parcel_area,unit_level,transaction_price,year,income_median_area,area_median_price
0,Condominium/Apartment,Kuala Lumpur Town Centre,PARK SEVEN,Freehold,4200.000000,8,4400000,2025,10655.952127,567500.0
1,Flat,Mukim Ampang,DESA PANDAN,Leasehold,721.181997,4,300000,2022,10231.924082,300000.0
2,Condominium/Apartment,Mukim Petaling,RESIDENSI HAMSTEAD,Leasehold,947.008837,16,411000,2024,10346.591315,375000.0
3,Condominium/Apartment,Kuala Lumpur Town Centre,PANGSAPURI BANDAR BARU SENTUL,Leasehold,559.723341,5,150000,2022,10302.979398,209000.0
4,Condominium/Apartment,Mukim Kuala Lumpur,PRISMA PERDANA APT,Freehold,818.057190,6,300000,2023,11404.069649,321000.0


In [33]:
X_test = test_set.drop(columns=['scheme_name_area', 'year', 'transaction_price'])
y_test = test_set['transaction_price']

In [69]:
y_hat = forest_best_model.predict(X_test)
test_error = root_mean_squared_error(y_test, y_hat)
print(f"Test RMSE: {test_error}")
MAPE = mean_absolute_percentage_error(y_test, y_hat)
print(f"Test MAPE: {MAPE}")

Test RMSE: 194129.61757052716
Test MAPE: 0.1592850652138078


In [35]:
X_test.head()

,property_type,mukim,tenure,land_parcel_area,unit_level,income_median_area,area_median_price
0,Condominium/Apartment,Kuala Lumpur Town Centre,Freehold,4200.000000,8,10655.952127,567500.0
1,Flat,Mukim Ampang,Leasehold,721.181997,4,10231.924082,300000.0
2,Condominium/Apartment,Mukim Petaling,Leasehold,947.008837,16,10346.591315,375000.0
3,Condominium/Apartment,Kuala Lumpur Town Centre,Leasehold,559.723341,5,10302.979398,209000.0
4,Condominium/Apartment,Mukim Kuala Lumpur,Freehold,818.057190,6,11404.069649,321000.0


In [103]:
y_test.quantile(0.50) # value is RM 470,000 when including training data

np.float64(480000.0)

In [104]:
y_test_below_50th = y_test[y_test <= 470000]
y_hat_below_50th = pd.Series(y_hat, index=y_test.index)
y_hat_below_50th = y_hat_below_50th[y_test <= 470000]
print(f"RMSE: {root_mean_squared_error(y_test_below_50th, y_hat_below_50th)}")
print(f"MAPE: {mean_absolute_percentage_error(y_test_below_50th, y_hat_below_50th)}")

RMSE: 67453.24723461336
MAPE: 0.17862584871275272


In [107]:
y_test.quantile(0.79) # percentile is 80th when including training data

np.float64(1000000.0)

In [85]:
y_test_below_1mil = y_test[y_test <= 1000000]
y_hat_below_1mil = pd.Series(y_hat, index=y_test.index)
y_hat_below_1mil = y_hat_below_1mil[y_test <= 1000000]
print(f"RMSE: {root_mean_squared_error(y_test_below_1mil, y_hat_below_1mil)}")
print(f"MAPE: {mean_absolute_percentage_error(y_test_below_1mil, y_hat_below_1mil)}")

RMSE: 113263.82330187809
MAPE: 0.16745165427422587


In [97]:
y_test.quantile(0.935)

np.float64(2000000.0)

In [99]:
below = 2000000
y_test_below = y_test[y_test <= below]
y_hat_below = pd.Series(y_hat, index=y_test.index)
y_hat_below = y_hat_below[y_test <= below]
print(f"RMSE: {root_mean_squared_error(y_test_below, y_hat_below)}")
print(f"MAPE: {mean_absolute_percentage_error(y_test_below, y_hat_below)}")

RMSE: 152402.97729035572
MAPE: 0.16247547787655212


In [100]:
y_test_above = y_test[y_test > 2000000]
y_hat_above = pd.Series(y_hat, index=y_test.index)
y_hat_above = y_hat_above[y_test > 2000000]
print(f"RMSE: {root_mean_squared_error(y_test_above, y_hat_above)}")
print(f"MAPE: {mean_absolute_percentage_error(y_test_above, y_hat_above)}")

RMSE: 501582.7943245399
MAPE: 0.1120898372585502


In [80]:
train_set.loc[[4]]

,property_type,mukim,scheme_name_area,tenure,land_parcel_area,unit_level,transaction_price,year,income_median_area,area_median_price
4,Condominium/Apartment,Mukim Setapak,GENTING COURT,Leasehold,832.050274,12,250000,2021,9519.480524,214000.0


In [81]:
sample = train_set.loc[[4]].drop(columns=['transaction_price', 'year'])
sample

,property_type,mukim,scheme_name_area,tenure,land_parcel_area,unit_level,income_median_area,area_median_price
4,Condominium/Apartment,Mukim Setapak,GENTING COURT,Leasehold,832.050274,12,9519.480524,214000.0


In [82]:
forest_best_model.predict(sample)

array([246837.66998283])

In [83]:
y_train.loc[[4]]

4    250000
Name: transaction_price, dtype: int64